# Krok 3. Okno historii N (E1)
**Wynik z instrukcji:** długość okna dla średniej kroczącej (zimny start, wariant porównawczy). Poziom: SKU. Dla wariantu `baseline_2` (jego „poziom bazowy”).

**Pytanie.** `baseline_2` zaczyna od poziomu sprzedaży towaru, czyli średniej z ostatnich N tygodni. Małe N szybko reaguje na zmiany, ale ma duży szum. Duże N jest stabilne, ale spóźnione i wymaga długiej historii (towar nowy jej nie ma: **zimny start**). Szukamy N, które daje najmniejszy błąd.

**Test (tylko lata EDA).** Co 4 tygodnie w 2024 udajemy, że prognozujemy kolejne 4 tygodnie wyłącznie z danych sprzed tej daty (ruchome okno prognozy). Sprawdzamy N = 4, 8, 13, 26, 52 tygodni. Typ popytu wyznaczamy z 2023.

**Wynik zapisany do:** `dane/eda/wynik_03_okno_N.csv` oraz `dane/eda/wynik_03_N_wg_typu.csv`.

In [5]:
# Ustawienia wspólne: moduł eda_wspolne.py leży obok tego notebooka
import os, sys
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import eda_wspolne as ew

ew.wejdz_do_korzenia()      # przechodzi do katalogu głównego projektu (tam, gdzie jest plik z danymi)
ew.styl()
print("Katalog projektu:", os.getcwd(), "| EDA używa danych do", ew.KONIEC_EDA.date(), "(rok testowy", ew.ROK_TEST, "pomijamy)")

Katalog projektu: /Users/mixer/python/analiza_sprzedazy_w_sklepie | EDA używa danych do 2024-12-31 (rok testowy 2025 pomijamy)


## Dane
#### Objaśnienie
**Po co.** Sprzedaż tygodniowa każdego towaru (tabela: towary × 104 tygodnie z lat 2023-2024) i typ popytu z 2023.
**Co robi kod.** Oprócz tabeli budujemy sumy skumulowane po tygodniach. Dzięki nim średnia z dowolnego okna to różnica dwóch liczb zamiast pętli po tygodniach, co przy tysiącach towarów jest wiele razy szybsze. Maska `akt` oznacza tygodnie od pierwszej sprzedaży towaru (wcześniejsze zera to brak towaru w ofercie, nie popyt).

In [6]:
fa = ew.wczytaj_sprzedaz()
kal = ew.kalendarz()
tyg = ew.panel_tygodniowy(fa, kal, "SKU")
del fa
assert tyg.shape[1] == 104, f"oczekiwano 104 tygodni, jest {tyg.shape[1]} ({tyg.columns[0].date()} … {tyg.columns[-1].date()})"
A = tyg.values.astype(float)
akt = ew.aktywne_tygodnie(tyg)
typ = ew.typ_popytu(tyg.iloc[:, :52])["typ"].values       # typ z 2023
C_sum = np.hstack([np.zeros((A.shape[0], 1)), np.cumsum(A * akt, axis=1)])     # C_sum[:, k] = suma pierwszych k tygodni
C_cnt = np.hstack([np.zeros((A.shape[0], 1)), np.cumsum(akt, axis=1)])         # liczba tygodni aktywnych
print("Towarów:", A.shape[0], "| typy:", pd.Series(typ).value_counts().to_dict())

Towarów: 9942 | typy: {'sporadyczny': 3553, 'za_malo_danych': 2832, 'regularny': 2578, 'nieregularny': 639, 'zmienny': 340}


## Ruchoma ocena okien N
#### Objaśnienie
**Po co.** Dla każdego N liczymy błąd prognozy „średnia z ostatnich N tygodni” na 2024.
**Co robi kod.**
- `t` to moment prognozy (indeks tygodnia). Okno obejmuje tygodnie `t-N … t-1`, czyli tylko przeszłość.
- Prognoza dla tygodni `t … t+3` jest stała: średnia z aktywnych tygodni okna (jeśli towar ma ich co najmniej `MIN_OBS`, inaczej brak prognozy).
- **WAPE** = suma |błąd| / suma sprzedaży, zsumowane po wszystkich momentach prognozy i towarach. Osobno dla każdego typu popytu.
- **okno_niepelne_%**: w jakiej części prognoz towar miał krócej niż N tygodni historii (to jest skala zimnego startu przy danym N).
- **pokrycie_%**: jaka część sprzedaży ma w ogóle prognozę (im większe N i `MIN_OBS`, tym więcej towarów nowych odpada).
**Jak czytać wynik.** Szukamy najniższego WAPE. Typowo dla regularnych towarów wygrywa średnie N, bo mały szum ma mniejsze znaczenie niż spóźnienie. Dla rzadkich im dłuższe okno, tym lepiej, bo krótkie okno ma za mało niezerowych tygodni.

In [7]:
NY = [4, 8, 13, 26, 52]
H, MIN_OBS = 4, 4
T0 = 52
momenty = list(range(T0, A.shape[1] - H + 1, H))
TYPY = ["regularny", "zmienny", "sporadyczny", "nieregularny", "za_malo_danych"]
blad = {N: {t: 0.0 for t in TYPY + ["wszystkie"]} for N in NY}
sprz = {N: {t: 0.0 for t in TYPY + ["wszystkie"]} for N in NY}
niepelne = {N: 0.0 for N in NY}; pary = {N: 0.0 for N in NY}; wolumen_calk = 0.0
for t in momenty:
    wolumen_calk += A[:, t:t + H].sum()
    for N in NY:
        s = max(0, t - N)
        cnt = C_cnt[:, t] - C_cnt[:, s]
        with np.errstate(all="ignore"):
            f = np.where(cnt >= MIN_OBS, (C_sum[:, t] - C_sum[:, s]) / cnt, np.nan)
        m = ~np.isnan(f)
        akt_t = A[:, t:t + H]
        e = np.abs(akt_t - f[:, None]); e[~m] = 0
        for tt in TYPY:
            mt = m & (typ == tt)
            blad[N][tt] += e[mt].sum(); sprz[N][tt] += akt_t[mt].sum()
        blad[N]["wszystkie"] += e[m].sum(); sprz[N]["wszystkie"] += akt_t[m].sum()
        niepelne[N] += (m & (cnt < N)).sum(); pary[N] += m.sum()

wyn = pd.DataFrame({N: {t: 100 * blad[N][t] / sprz[N][t] if sprz[N][t] > 0 else np.nan for t in TYPY + ["wszystkie"]} for N in NY}).T
wyn.index.name = "N_tygodni"
wyn["okno_niepelne_%"] = [100 * niepelne[N] / pary[N] for N in NY]
wyn["pokrycie_%"] = [100 * sprz[N]["wszystkie"] / wolumen_calk for N in NY]
print("WAPE (%) wg okna N i typu popytu:")
print(wyn.round(1).to_string())

WAPE (%) wg okna N i typu popytu:
           regularny  zmienny  sporadyczny  nieregularny  za_malo_danych  wszystkie  okno_niepelne_%  pokrycie_%
N_tygodni                                                                                                       
4               41.7     62.9         90.2         104.8            85.0       52.5              0.0        97.6
8               40.9     63.3         90.6         108.2            91.7       52.6              2.0        97.6
13              41.4     65.3         93.8         112.4            99.4       54.1              4.6        97.6
26              43.3     70.0        103.1         124.8           112.8       58.1             11.3        97.6
52              45.0     71.5        119.2         142.4           121.1       62.0             27.5        97.6


## Wniosek i zapis
#### Objaśnienie
**Po co.** Wybieramy N dla każdego typu i ogółem. To parametr `TYGODNI_POZIOM` dla `baseline_2`.
**Jak czytać wynik.** Wybór „minimalny WAPE” bywa płaski: gdy różnica między sąsiednimi N jest mała (poniżej 1 punktu procentowego), wybieramy **krótsze** okno, bo szybciej reaguje na zmiany i szybciej uruchamia prognozę dla nowych towarów. Towary `za_malo_danych` (zimny start) dostają inną regułę w kroku 8 i 9: tu pokazujemy tylko, jak bardzo wypadają gorzej.

In [8]:
def wybierz(kol):
    s = wyn[kol].dropna()
    if s.empty:
        return np.nan
    return int(s.index[(s <= s.min() + 1.0).values][0])           # najkrótsze okno w granicach 1 pp od najlepszego

wybor = pd.Series({t: wybierz(t) for t in TYPY + ["wszystkie"]}, name="N_wybrane")
print(wybor.to_string())
ew.zapisz_wynik(wyn, "wynik_03_okno_N.csv")
ew.zapisz_wynik(wybor.to_frame(), "wynik_03_N_wg_typu.csv")

regularny         4
zmienny           4
sporadyczny       4
nieregularny      4
za_malo_danych    4
wszystkie         4
Zapisano: dane/eda/wynik_03_okno_N.csv (5 wierszy)
Zapisano: dane/eda/wynik_03_N_wg_typu.csv (6 wierszy)


'dane/eda/wynik_03_N_wg_typu.csv'